# Objective:

This project aims to use deep learning technologies to recognize the emotions from voice interactions. As a possible extra activity in the project, We would like to explore the possibility of having a model trained on a single language dataset being tested for multiple languages, understanding how predicted emotions compare in feature information and overall similarity.


## Project Background: 

Speech Emotion Recognition can be an incredible tool, used during interaction between human to human (H2H), human to machine(H2M), and Machine to Machine (M2M),  attempting to recognize emotion and affective states from speech. 

Human voice, through tone, pitch and velocity, often reflects underlying emotions and can convey a lot of context during interactions. Even animals are equipped with capabilities to recognize emotions from voice and general human expressions. 

As part of speech recognition, this aspect is incredibly important for systems that can deal with humans in a variety of situations: 
telemarketing and customer service
shopping assistants
emergency response 
patient care and health advice

Analyzing voice features to identify changes in emotions can be an important tool to prevent interactions from escalating, allowing a better understanding of situational context that can impact action planning in professional services settings.


### Import Libraries and Install Dependencies

In [16]:
# install dependencies
!pip install -q --disable-pip-version-check awswrangler pyathena
!pip install -q --upgrade boto3 botocore awscli
!pip install -q librosa soundfile audioread

In [17]:
import boto3
import sagemaker
import pandas as pd
import awswrangler as wr
from pyathena import connect
import math
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from datasets import load_dataset

### Load in Data and Store

In [3]:
# define dataset in s3
S3_ZIP = "s3://ser-multilingual-spr2026msaai540-group2/CREMAD.zip"

In [4]:
# store zip file and extract data
ZIP_LOCAL = Path.home() / "datasets" / "zips" / "CREMAD.zip"
DATA_ROOT = Path.home() / "datasets" / "CREMAD"
AUDIO_DIR = DATA_ROOT / "AudioWAV"

In [5]:
# confirm folders exist
ZIP_LOCAL.parent.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

In [6]:
# download zip if needed
if not ZIP_LOCAL.exists():
    !aws s3 cp {S3_ZIP} {str(ZIP_LOCAL)}
else:
    print(f"Zip already exists: {ZIP_LOCAL}")

Zip already exists: /home/sagemaker-user/datasets/zips/CREMAD.zip


In [7]:
# unzip if AudioWav doesn't exist
if not AUDIO_DIR.exists() or len(list(AUDIO_DIR.glob("*.wav"))) == 0:
    !unzip -q {str(ZIP_LOCAL)} -d {str(DATA_ROOT)}
else:
    print(f"Audio already extracted: {AUDIO_DIR} (wav files found)")

print("AUDIO_DIR =", str(AUDIO_DIR))

Audio already extracted: /home/sagemaker-user/datasets/CREMAD/AudioWAV (wav files found)
AUDIO_DIR = /home/sagemaker-user/datasets/CREMAD/AudioWAV


### Audio File Processing before Modeling

In [8]:
# create metadata from file names
AUDIO_DIR = Path(AUDIO_DIR)

# CREMA-D emotion map
EMOTION_MAP = {
    "ANG": "anger",
    "DIS": "disgust",
    "FEA": "fear",
    "HAP": "happy",
    "NEU": "neutral",
    "SAD": "sad",
}

# file name pattern
pattern = re.compile(
    r"^(?P<speaker_id>\d+)_"
    r"(?P<utterance>[A-Z]{2,4})_"
    r"(?P<emotion_code>[A-Z]{3})_"
    r"(?P<intensity>[A-Z]{1,2})"
    r"\.wav$",
    re.IGNORECASE
)

rows = []
bad = []

# extract speaker id, emotion, and utterance from audio filename
for fp in sorted(AUDIO_DIR.glob("*.wav")):
    m = pattern.match(fp.name)
    if not m:
        bad.append(fp.name)
        continue

    d = m.groupdict()
    emotion_code = d["emotion_code"].upper()

    rows.append({
        "filepath": str(fp),
        "filename": fp.name,
        "speaker_id": int(d["speaker_id"]),
        "utterance": d["utterance"].upper(),
        "emotion_code": emotion_code,
        "emotion": EMOTION_MAP.get(emotion_code, "unknown"),
        "intensity": d["intensity"].upper(),
    })

meta = pd.DataFrame(rows)

print("Parsed rows:", len(meta))
print("Unparsed filenames:", len(bad))
if bad:
    print("Examples of unparsed filenames:", bad[:10])

meta.head()

Parsed rows: 7442
Unparsed filenames: 0


,filepath,filename,speaker_id,utterance,emotion_code,emotion,intensity
0,/home/sagemaker-user/datasets/CREMAD/AudioWAV/...,1001_DFA_ANG_XX.wav,1001,DFA,ANG,anger,XX
1,/home/sagemaker-user/datasets/CREMAD/AudioWAV/...,1001_DFA_DIS_XX.wav,1001,DFA,DIS,disgust,XX
2,/home/sagemaker-user/datasets/CREMAD/AudioWAV/...,1001_DFA_FEA_XX.wav,1001,DFA,FEA,fear,XX
3,/home/sagemaker-user/datasets/CREMAD/AudioWAV/...,1001_DFA_HAP_XX.wav,1001,DFA,HAP,happy,XX
4,/home/sagemaker-user/datasets/CREMAD/AudioWAV/...,1001_DFA_NEU_XX.wav,1001,DFA,NEU,neutral,XX


Instead of splitting the dataset by random files, we'll split by speakers to ensure the model can generalize on emotion and speakers.

In [11]:
# to ensure our model can generalize on different speakers, we'll perform a speaker split
# this prevents the model from learning from the speaker vs emotions

RANDOM_SEED = 100
np.random.seed(RANDOM_SEED)

# define unique speakers
unique_speakers = meta["speaker_id"].unique()
num_speakers = len(unique_speakers)

print(f"distinct speakers: {num_speakers}")

# shuffle speakers
shuffled_speakers = np.random.permutation(unique_speakers)

# split speakers 80/10/10 for train/validation/test
train_end = int(0.8 * num_speakers)
val_end = int(0.9 * num_speakers)

train_speakers = set(shuffled_speakers[:train_end])
val_speakers   = set(shuffled_speakers[train_end:val_end])
test_speakers  = set(shuffled_speakers[val_end:])

print(f"Train speakers: {len(train_speakers)}")
print(f"Val speakers:   {len(val_speakers)}")
print(f"Test speakers:  {len(test_speakers)}")

# define which speakers are train/validation/test
def assign_split(speaker_id):
    if speaker_id in train_speakers:
        return "train"
    elif speaker_id in val_speakers:
        return "val"
    else:
        return "test"

meta["split"] = meta["speaker_id"].apply(assign_split)

distinct speakers: 91
Train speakers: 72
Val speakers:   9
Test speakers:  10


In [12]:
# speaker distribution between splits
pd.crosstab(meta["split"], meta["emotion_code"], normalize="index")

emotion_code,ANG,DIS,FEA,HAP,NEU,SAD
split,,,,,,
test,0.170762,0.170762,0.170762,0.170762,0.146192,0.170762
train,0.170798,0.170798,0.170798,0.170798,0.146010,0.170798
val,0.170732,0.170732,0.170732,0.170732,0.146341,0.170732


### Light Weight EDA

In [13]:
# count speaker clips
speaker_clip_counts = (
    meta.groupby(["split", "speaker_id"])
        .size()
        .reset_index(name="num_clips")
)

speaker_clip_counts.groupby("split")["num_clips"].describe()

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,10.0,81.400000,1.897367,76.0,82.0,82.0,82.0,82.0
train,72.0,81.805556,1.001954,76.0,82.0,82.0,82.0,82.0
val,9.0,82.000000,0.000000,82.0,82.0,82.0,82.0,82.0


We want to ensure that the median and mean clips per speaker are close

### Extract MFCC (Mel-frequency cepstral coefficients) features

In [21]:
label_col = "emotion_code"

def extract_audio_features(file_path, n_mfcc=40, max_len=200):
    audio, sample_rate = librosa.load(file_path, sr=None)
    audio_trimmed, _ = librosa.effects.trim(audio, top_db=30)  # same as your local

    def pad_or_truncate(feature, max_len):
        return (np.pad(feature, ((0, 0), (0, max_len - feature.shape[1])), mode='constant')
                if feature.shape[1] < max_len else feature[:, :max_len])

    mfcc = librosa.feature.mfcc(y=audio_trimmed, sr=sample_rate, n_mfcc=n_mfcc)
    mfcc = pad_or_truncate(mfcc, max_len)
    return mfcc

def build_split_arrays(split_name):
    features = []
    labels = []

    df = meta.loc[meta["split"] == split_name, ["filepath", label_col]].reset_index(drop=True)

    for i, row in df.iterrows():
        fp = row["filepath"]
        lab = row[label_col]

        mfcc = extract_audio_features(fp, n_mfcc=40, max_len=200)
        features.append(mfcc)
        labels.append(lab)

        if (i + 1) % 500 == 0:
            print(f"{split_name}: {i+1}/{len(df)}")

    X = np.array(features)
    y = np.array(labels)
    return X, y

X_train, y_train = build_split_arrays("train")
X_val,   y_val   = build_split_arrays("val")
X_test,  y_test  = build_split_arrays("test")

print("Dataset shapes:")
print("train", X_train.shape, y_train.shape)
print("validation", X_val.shape, y_val.shape)
print("test", X_test.shape, y_test.shape)

train: 500/5890
train: 1000/5890
train: 1500/5890
train: 2000/5890
train: 2500/5890
train: 3000/5890
train: 3500/5890
train: 4000/5890
train: 4500/5890
train: 5000/5890
train: 5500/5890
val: 500/738
test: 500/814
Dataset shapes:
train (5890, 40, 200) (5890,)
validation (738, 40, 200) (738,)
test (814, 40, 200) (814,)
